# BigBasket FMCG — Exploratory Data Analysis
**Dataset:** 27,555 SKUs · 11 categories · 90 sub-categories  
**Source:** [Kaggle](https://www.kaggle.com/datasets/surajjha101/bigbasket-entire-product-list-28k-datapoints)  
**GitHub:** [goodatchess/bigbasket-eda-dashboard](https://github.com/goodatchess/bigbasket-eda-dashboard)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

os.makedirs('plots', exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})
print("Libraries loaded.")

## 1. Load & Inspect

In [ ]:
# Download CSV from Kaggle, rename to bigbasket.csv, place in this folder
df = pd.read_csv('bigbasket.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 2. Data Cleaning

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Columns:", df.columns.tolist())

df = df.dropna(subset=['sale_price', 'market_price'])
df = df[df['sale_price'] <= df['market_price']]

df['discount_pct'] = ((df['market_price'] - df['sale_price']) / df['market_price'] * 100).round(2)

def price_tier(p):
    if p < 100:   return 'Budget (<Rs100)'
    elif p < 500: return 'Mid (Rs100-500)'
    else:         return 'Premium (Rs500+)'

df['price_tier'] = df['sale_price'].apply(price_tier)
print(f"Clean dataset: {df.shape[0]:,} rows")
df[['sale_price', 'market_price', 'discount_pct']].describe().round(2)

## 3. Platform-Level KPIs

In [ ]:
print(f"Total SKUs       : {len(df):,}")
print(f"Avg MRP          : Rs{df['market_price'].mean():.2f}")
print(f"Avg Sale Price   : Rs{df['sale_price'].mean():.2f}")
print(f"Avg Discount     : {df['discount_pct'].mean():.1f}%")
if 'rating' in df.columns:
    print(f"Avg Rating       : {df['rating'].mean():.2f} / 5.0")
print(f"Categories       : {df['category'].nunique() if 'category' in df.columns else 'N/A'}")
print(f"Sub-categories   : {df['sub_category'].nunique() if 'sub_category' in df.columns else 'N/A'}")

## 4. SKU Count by Category

In [ ]:
cat_col = 'category'
sku_by_cat = df[cat_col].value_counts().reset_index()
sku_by_cat.columns = ['category', 'sku_count']
sku_by_cat['pct'] = (sku_by_cat['sku_count'] / sku_by_cat['sku_count'].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sku_by_cat['category'], sku_by_cat['sku_count'],
               color=sns.color_palette("Blues_d", len(sku_by_cat)))
ax.bar_label(bars, labels=[f"{p}%" for p in sku_by_cat['pct']], padding=4, fontsize=9)
ax.set_xlabel("Number of SKUs")
ax.set_title("SKU Count by Category", fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('plots/sku_by_category.png', bbox_inches='tight')
plt.show()
print("Top 3 categories:", round(sku_by_cat.head(3)['pct'].sum(), 1), "% of catalogue")

## 5. Discount Depth by Category

In [ ]:
disc_by_cat = df.groupby(cat_col)['discount_pct'].mean().sort_values(ascending=False).reset_index()
disc_by_cat.columns = ['category', 'avg_discount']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(disc_by_cat['category'], disc_by_cat['avg_discount'],
               color=sns.color_palette("RdYlGn_r", len(disc_by_cat)))
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in disc_by_cat['avg_discount']], padding=4, fontsize=9)
ax.set_xlabel("Avg Discount %")
ax.set_title("Average Discount Depth by Category", fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('plots/discount_by_category.png', bbox_inches='tight')
plt.show()

## 6. Price Tier Distribution

In [ ]:
tier_counts = df['price_tier'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', startangle=140,
            wedgeprops=dict(width=0.55), colors=sns.color_palette("Set2", len(tier_counts)))
axes[0].set_title("SKU Split by Price Tier", fontweight='bold')

axes[1].hist(df[df['sale_price'] <= 2000]['sale_price'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_xlabel("Sale Price (Rs)")
axes[1].set_ylabel("SKU Count")
axes[1].set_title("Sale Price Distribution", fontweight='bold')

plt.tight_layout()
plt.savefig('plots/price_distribution.png', bbox_inches='tight')
plt.show()

## 7. Average Rating by Category

In [ ]:
if 'rating' in df.columns:
    rating_by_cat = df.groupby(cat_col)['rating'].mean().sort_values(ascending=False).reset_index()
    rating_by_cat.columns = ['category', 'avg_rating']
    colors = ['#2ecc71' if r >= 4.0 else '#e67e22' if r >= 3.8 else '#e74c3c' for r in rating_by_cat['avg_rating']]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(rating_by_cat['category'], rating_by_cat['avg_rating'], color=colors)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in rating_by_cat['avg_rating']], padding=4, fontsize=9)
    ax.set_xlim(0, 5)
    ax.set_xlabel("Avg Rating (out of 5)")
    ax.set_title("Average Customer Rating by Category", fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('plots/ratings_by_category.png', bbox_inches='tight')
    plt.show()
    print("Highest rated:", rating_by_cat.iloc[0]['category'], f"({rating_by_cat.iloc[0]['avg_rating']:.2f})")
    print("Lowest  rated:", rating_by_cat.iloc[-1]['category'], f"({rating_by_cat.iloc[-1]['avg_rating']:.2f})")
else:
    print("No rating column found.")

## 8. Top 15 Brands by SKU Count

In [ ]:
brand_col = 'brand'
if brand_col in df.columns:
    top_brands = df[brand_col].value_counts().head(15).reset_index()
    top_brands.columns = ['brand', 'sku_count']
    brand_disc = df.groupby(brand_col)['discount_pct'].mean().reset_index()
    brand_disc.columns = ['brand', 'avg_discount']
    top_brands = top_brands.merge(brand_disc, on='brand')

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top_brands['brand'], top_brands['sku_count'],
                   color=sns.color_palette("viridis", len(top_brands)))
    ax.bar_label(bars, labels=[f"{v} SKUs ({d:.1f}% disc)"
                  for v, d in zip(top_brands['sku_count'], top_brands['avg_discount'])], padding=4, fontsize=8)
    ax.set_xlabel("Number of SKUs")
    ax.set_title("Top 15 Brands by Catalogue Depth", fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('plots/top_brands.png', bbox_inches='tight')
    plt.show()
else:
    print(f"Column '{brand_col}' not found. Columns:", df.columns.tolist())

## 9. Discount vs Rating Correlation

In [ ]:
if 'rating' in df.columns:
    sample = df[['discount_pct', 'rating']].dropna().sample(min(3000, len(df)), random_state=42)
    corr = sample.corr().loc['discount_pct', 'rating']

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(sample['discount_pct'], sample['rating'], alpha=0.15, s=10, color='steelblue')
    m, b = np.polyfit(sample['discount_pct'], sample['rating'], 1)
    x_line = np.linspace(sample['discount_pct'].min(), sample['discount_pct'].max(), 100)
    ax.plot(x_line, m * x_line + b, color='tomato', linewidth=1.5, label=f'Trend (r={corr:.2f})')
    ax.set_xlabel("Discount %")
    ax.set_ylabel("Rating")
    ax.set_title("Discount % vs Customer Rating", fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('plots/discount_vs_rating.png', bbox_inches='tight')
    plt.show()
    print(f"Pearson r = {corr:.3f}")
    if abs(corr) < 0.1:   print("Near-zero — discount has little effect on rating.")
    elif corr < 0:         print("Weak negative — heavily discounted products rate slightly lower.")
    else:                  print("Positive — discounted products rate higher.")
else:
    print("No rating column.")

## 10. Key Findings

| # | Finding |
|---|---------|
| 1 | **Beauty & Hygiene** leads SKU count (28.7%) but has the lowest avg rating — suggests over-stocking of lower-quality products |
| 2 | **Eggs, Meat & Fish** has the highest avg rating with the lowest discount — customers buy it without needing a price cut |
| 3 | **~86% of SKUs are priced under Rs500** — catalogue is heavily mass-market focused |
| 4 | Platform-wide avg discount is **~15%** — premium categories discount less |
| 5 | Discount % and rating show **near-zero correlation** — quality perception is independent of pricing strategy |

---
*GitHub: [goodatchess/bigbasket-eda-dashboard](https://github.com/goodatchess/bigbasket-eda-dashboard)*


## 11. Supply Chain Intelligence
Simulated inventory parameters derived from catalogue velocity and discount signals.
Covers: Safety Stock, Reorder Point, Days of Supply, Inventory Turnover, Stockout Risk.

**Formulas used:**
- `Safety Stock = Z × σ(demand) × √(Lead Time)`  — Z=1.65 for 95% service level
- `Reorder Point = (Avg Daily Demand × Lead Time) + Safety Stock`
- `Days of Supply = Current Stock / Avg Daily Demand`
- `Inventory Turnover = 365 / Days of Supply`


In [ ]:
import numpy as np

# Build category-level SC metrics
cat_stats = df.groupby('category').agg(
    sku_count    = ('sale_price', 'count'),
    avg_discount = ('discount_pct', 'mean'),
    avg_price    = ('sale_price', 'mean'),
).reset_index()

if 'rating' in df.columns:
    cat_stats = cat_stats.merge(
        df.groupby('category')['rating'].mean().reset_index().rename(columns={'rating':'avg_rating'}),
        on='category', how='left'
    )

np.random.seed(42)

# Demand velocity score (0-1): SKU breadth + discount depth proxy
cat_stats['velocity_score'] = (
    (cat_stats['sku_count'] / cat_stats['sku_count'].max()) * 0.6 +
    (cat_stats['avg_discount'] / cat_stats['avg_discount'].max()) * 0.4
).round(3)

# Simulated avg daily demand (50-500 units, velocity-driven)
cat_stats['avg_daily_demand'] = (50 + cat_stats['velocity_score'] * 450).round(0).astype(int)

# Demand std dev
cat_stats['demand_std'] = (cat_stats['avg_daily_demand'] * 0.15 + cat_stats['avg_discount'] * 0.8).round(1)

# Lead time (days): premium items take longer
cat_stats['lead_time_days'] = np.where(cat_stats['avg_price'] > 500, 5,
                               np.where(cat_stats['avg_price'] > 100, 3, 2))

# Safety Stock = Z * sigma * sqrt(LT)
Z = 1.65
cat_stats['safety_stock'] = (Z * cat_stats['demand_std'] * np.sqrt(cat_stats['lead_time_days'])).round(0).astype(int)

# Reorder Point
cat_stats['reorder_point'] = (cat_stats['avg_daily_demand'] * cat_stats['lead_time_days'] + cat_stats['safety_stock']).astype(int)

# Simulated current stock
cat_stats['current_stock'] = (cat_stats['reorder_point'] * np.random.uniform(0.6, 2.2, len(cat_stats))).round(0).astype(int)

# Days of Supply
cat_stats['days_of_supply'] = (cat_stats['current_stock'] / cat_stats['avg_daily_demand']).round(1)

# Inventory Turnover
cat_stats['inventory_turnover'] = (365 / cat_stats['days_of_supply']).round(1)

# Stockout Risk Flag
def risk_flag(row):
    if row['current_stock'] <= row['safety_stock']:     return 'Critical'
    elif row['current_stock'] <= row['reorder_point']:  return 'Reorder Now'
    elif row['days_of_supply'] < 5:                     return 'Low Stock'
    else:                                               return 'Healthy'

cat_stats['stock_status'] = cat_stats.apply(risk_flag, axis=1)

print('SC Metrics built:')
print(cat_stats[['category','avg_daily_demand','safety_stock','reorder_point',
                  'current_stock','days_of_supply','inventory_turnover','stock_status']].to_string(index=False))


In [ ]:
# Plot 1: Days of Supply by Category
dos = cat_stats.set_index('category')['days_of_supply'].sort_values()
colors = ['#e74c3c' if v < 3 else '#e67e22' if v < 6 else '#2ecc71' for v in dos.values]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(dos.index, dos.values, color=colors)
ax.bar_label(bars, labels=[f'{v:.1f}d' for v in dos.values], padding=4, fontsize=9)
ax.axvline(3, color='#e74c3c', lw=1.5, ls='--', label='Critical threshold (3d)')
ax.axvline(6, color='#e67e22', lw=1.5, ls='--', label='Low stock threshold (6d)')
ax.set_xlabel('Days of Supply')
ax.set_title('Days of Supply by Category', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('plots/sc_days_of_supply.png', bbox_inches='tight')
plt.show()


In [ ]:
# Plot 2: Inventory Turnover
it = cat_stats.set_index('category')['inventory_turnover'].sort_values(ascending=False)
colors_it = ['#2ecc71' if v >= 12 else '#e67e22' if v >= 6 else '#e74c3c' for v in it.values]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(it.index, it.values, color=colors_it)
ax.bar_label(bars, labels=[f'{v:.1f}x' for v in it.values], padding=4, fontsize=9)
ax.axvline(8, color='#2ecc71', lw=1.5, ls='--', label='FMCG benchmark (8x)')
ax.set_xlabel('Inventory Turnover (times/year)')
ax.set_title('Inventory Turnover by Category', fontsize=13, fontweight='bold')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('plots/sc_inventory_turnover.png', bbox_inches='tight')
plt.show()


In [ ]:
# Plot 3: Current Stock vs Reorder Point
cs = cat_stats.sort_values('days_of_supply')
x = np.arange(len(cs))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(x - 0.2, cs['current_stock'], height=0.35, color='#185fa5', label='Current Stock')
ax.barh(x + 0.2, cs['reorder_point'], height=0.35, color='#e67e22', label='Reorder Point', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(cs['category'], fontsize=9)
ax.set_xlabel('Units')
ax.set_title('Current Stock vs Reorder Point by Category', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('plots/sc_stock_vs_rop.png', bbox_inches='tight')
plt.show()


In [ ]:
# Plot 4: Velocity vs DOS Scatter (Stockout Risk Map)
color_map = {'Critical': '#e74c3c', 'Reorder Now': '#f39c12', 'Low Stock': '#e67e22', 'Healthy': '#2ecc71'}

fig, ax = plt.subplots(figsize=(10, 6))
for status, grp in cat_stats.groupby('stock_status'):
    ax.scatter(grp['velocity_score'], grp['days_of_supply'],
               color=color_map.get(status, '#aaa'), label=status, s=120, edgecolors='white', lw=0.5)
    for _, row in grp.iterrows():
        ax.annotate(row['category'][:14], (row['velocity_score'], row['days_of_supply']),
                    textcoords='offset points', xytext=(5, 4), fontsize=8)
ax.axhline(3, color='#e74c3c', lw=1.2, ls='--', alpha=0.7, label='Critical DOS threshold')
ax.set_xlabel('Demand Velocity Score')
ax.set_ylabel('Days of Supply')
ax.set_title('Stockout Risk Map — Velocity vs Days of Supply', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('plots/sc_risk_map.png', bbox_inches='tight')
plt.show()


## 12. Supply Chain Key Findings

| # | Finding |
|---|---------|
| 1 | Categories with **high velocity + low DOS** are at immediate stockout risk — priority replenishment needed |
| 2 | **Safety stock** calculated at 95% service level (Z=1.65) — buffers against demand variability and lead time uncertainty |
| 3 | FMCG benchmark inventory turnover is **8–12x/year** — categories below this have working capital tied up unnecessarily |
| 4 | **Reorder Point = (Avg Daily Demand × Lead Time) + Safety Stock** — trigger POs when stock hits this level |
| 5 | High-discount categories show **higher demand variability** — requiring larger safety stock buffers |

---
*GitHub: [goodatchess/bigbasket-eda-dashboard](https://github.com/goodatchess/bigbasket-eda-dashboard)*
